# GLOSSATE CUDA Colab Template

Template for running GLOSSATE in Google Colab with CUDA. Set **Runtime > Change runtime type > GPU** before running the notebook.

This template keeps the public GLOSSATE API calls (`glossate.subtitle`, `glossate.transcribe`, `glossate.translate`, `glossate.write`, `glossate.Session`) but swaps the Colab runtime to CUDA-backed models instead of MLX.

## 1. Install dependencies

Use the PyPI package by default. To test a branch or fork, change `GLOSSATE_INSTALL` to a Git URL such as `git+https://github.com/anaxoniclabs/GLOSSATE.git@main`.

In [ ]:
GLOSSATE_INSTALL = "glossate"

!apt-get -qq update
!apt-get -qq install -y ffmpeg
!python -m pip install -q --upgrade pip
!python -m pip install -q "{GLOSSATE_INSTALL}[cuda,detect]"
# Gemma 4 is recent; make sure transformers/accelerate are current.
!python -m pip install -q --upgrade "transformers>=5.5.0" accelerate

## 1b. Authenticate for gated Gemma weights

Gemma models on Hugging Face are gated: accept the license on the model page (e.g. [`google/gemma-4-E4B-it`](https://huggingface.co/google/gemma-4-E4B-it)) and log in with a token that has access. Skip this if you only use NLLB models.

In [ ]:
from huggingface_hub import login

# Paste a token from https://huggingface.co/settings/tokens (or set HF_TOKEN).
login()  # interactive; or login(token="hf_...")

## 2. Check CUDA

In [ ]:
import torch

assert torch.cuda.is_available(), "CUDA is not available. In Colab, choose Runtime > Change runtime type > GPU."
print("CUDA:", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(0))

!nvidia-smi

## 3. Configure input and output paths

Enter a Colab path, for example `/content/video.mp4` or `/content/drive/MyDrive/video.mp4`. If you want Drive paths, set `USE_GOOGLE_DRIVE = True`.

In [ ]:
USE_GOOGLE_DRIVE = False

if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")

# Required: path to the input audio/video file.
INPUT_PATH = "/content/your-video-or-audio-file.mp4"

# Optional: leave SOURCE_LANG = None to auto-detect. Use ISO 639-1 codes like "en", "tr", "ar".
SOURCE_LANG = None
TARGET_LANG = "tr"

# ASR models accepted by the GLOSSATE API: tiny, base, small, medium, large, turbo.
# On CUDA they map to faster-whisper and run through BatchedInferencePipeline.
ASR_MODEL = "turbo"

# Translation uses Gemma 4 (via HF transformers) on CUDA. General multilingual.
# Options by VRAM: gemma-4-e2b, gemma-4-e4b (default, ~16GB),
#   gemma-4-26b (MoE, ~52GB), gemma-4-31b (~62GB, A100 80GB).
# You can also pass an NLLB model (nllb-200-600m / 1.3b / 3.3b) instead.
MT_MODEL = "gemma-4-e4b"

OUTPUT_FORMAT = "srt"
OUTPUT_PATH = "/content/glossate-output.srt"

Optional upload helper. Run this cell only if you want to upload a file directly into the Colab session instead of using Drive.

In [ ]:
# from google.colab import files
# uploaded = files.upload()
# INPUT_PATH = "/content/" + next(iter(uploaded.keys()))
# print(INPUT_PATH)

## 4. Configure CUDA backends

GLOSSATE now uses the CUDA backends directly: `faster-whisper` for ASR and NLLB via PyTorch/Transformers for translation.

In [ ]:
from pathlib import Path

import glossate

ASR_BACKEND = "faster-whisper"
MT_BACKEND = "transformers"   # Gemma 4 on CUDA; use "nllb" for NLLB models
# "auto" => float16 for faster-whisper ASR and bfloat16 for the Gemma LLM.
COMPUTE_TYPE = "auto"

print("GLOSSATE CUDA backends configured.")

## 5. Run the full subtitle pipeline

This is the simplest path: extract audio if needed, transcribe, translate, and write the subtitle file.

In [ ]:
input_path = Path(INPUT_PATH)
assert input_path.exists(), f"File not found: {input_path}"

output = glossate.subtitle(
    input_path,
    source=SOURCE_LANG,
    target=TARGET_LANG,
    asr_model=ASR_MODEL,
    mt_model=MT_MODEL,
    format=OUTPUT_FORMAT,
    output=OUTPUT_PATH,
    device="cuda",
    asr_backend=ASR_BACKEND,
    mt_backend=MT_BACKEND,
    compute_type=COMPUTE_TYPE,
)

print("Wrote:", output)

## 6. Step-by-step API usage

Use this when you want to inspect or edit cues between transcription and translation.

In [ ]:
cues = glossate.transcribe(INPUT_PATH, model=ASR_MODEL, source=SOURCE_LANG, device="cuda", backend=ASR_BACKEND, compute_type=COMPUTE_TYPE)
cues[:3]

In [ ]:
detected_source = SOURCE_LANG or (cues[0].lang if cues else None)
translated = glossate.translate(cues, target=TARGET_LANG, source=detected_source, model=MT_MODEL, device="cuda", backend=MT_BACKEND, compute_type=COMPUTE_TYPE)
translated[:3]

In [ ]:
glossate.write(translated, OUTPUT_PATH)

## 7. Batch template

For multiple files, `Session` loads the CUDA ASR and translation models once and reuses them.

In [ ]:
FILES = [
    "/content/your-video-or-audio-file.mp4",
]

with glossate.Session(asr_model=ASR_MODEL, mt_model=MT_MODEL, device="cuda", asr_backend=ASR_BACKEND, mt_backend=MT_BACKEND, compute_type=COMPUTE_TYPE) as session:
    for file_path in FILES:
        file_path = Path(file_path)
        out_path = file_path.with_suffix(f".{TARGET_LANG}.{OUTPUT_FORMAT}")
        written = session.subtitle(
            file_path,
            source=SOURCE_LANG,
            target=TARGET_LANG,
            format=OUTPUT_FORMAT,
            output=out_path,
        )
        print(written)

## 8. Download result

In [ ]:
from google.colab import files

files.download(str(OUTPUT_PATH))